## Checkpoint → trace_pairs → Neo4j

In [1]:
import sys, json
from pathlib import Path
from dotenv import load_dotenv

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

TRAIN_JSONL = ROOT / "train.jsonl"
MAX_LINES = 1090
MAX_REQUIREMENTS = 2000
DOC_ID = "train_jsonl"
CHECKPOINT_PATH = ROOT / "requirements_checkpoint_embed_tier.json"
LOAD_CHECKPOINT = True

In [2]:
from src.schema import Entry

if LOAD_CHECKPOINT and CHECKPOINT_PATH.exists():
    data = json.loads(CHECKPOINT_PATH.read_text(encoding="utf-8"))
    requirements_with_tiers = [(Entry(**d["entry"]), d["tier"]) for d in data["requirements_with_tiers"]]
    trace_pairs = [tuple(p) for p in data["trace_pairs"]]
    requirements = [e for e, _ in requirements_with_tiers]
    DOC_ID = data.get("doc_id", DOC_ID)
    emb = data.get("embeddings") or []
    embeddings = emb if len(emb) == len(requirements_with_tiers) else None
    print(f"Loaded checkpoint: {len(requirements_with_tiers)} requirements, {len(trace_pairs)} trace pairs, embeddings={'yes' if embeddings else 'no'}.")
else:
    requirements = requirements_with_tiers = trace_pairs = embeddings = None

Loaded checkpoint: 2000 requirements, 1540 trace pairs, embeddings=yes.


Optional: recompute trace_pairs

In [3]:
from src.trace_similarity_nova import get_trace_pairs_single_choice_nova_with_fallback

trace_pairs = get_trace_pairs_single_choice_nova_with_fallback(requirements_with_tiers, embeddings, k=5, nova_batch_size=5)
print(f"Trace pairs (one per system/component): {len(trace_pairs)}")

Trace pairs (one per system/component): 1540


In [4]:
from src.neo4j_loader import (
    get_driver,
    load_tiered_requirements_into_neo4j,
    requirement_full_id,
    delete_trace_edges,
    create_trace_edges,
    filter_trace_pairs_to_adjacent_tiers,
)

driver = get_driver()
n = load_tiered_requirements_into_neo4j(requirements_with_tiers, driver=driver, document_id=DOC_ID, embeddings=embeddings if embeddings else None)
print(f"Loaded {n} requirement nodes.")

full_ids = [requirement_full_id(DOC_ID, e, i) for i, (e, _) in enumerate(requirements_with_tiers)]
deleted = delete_trace_edges(driver)
if deleted:
    print(f"Deleted {deleted} existing TRACES_TO edges.")
all_pairs = filter_trace_pairs_to_adjacent_tiers(requirements_with_tiers, trace_pairs)
edges = create_trace_edges(full_ids, all_pairs, driver=driver)
print(f"Created {edges} TRACES_TO edges ({len(all_pairs)} pairs).")
driver.close()

Loaded 2000 requirement nodes.
Created 3080 TRACES_TO edges (1540 pairs).


In [5]:
checkpoint = {
    "requirements_with_tiers": [{"entry": e.model_dump(), "tier": t} for e, t in requirements_with_tiers],
    "trace_pairs": [list(p) for p in trace_pairs],
    "doc_id": DOC_ID,
}
if embeddings and len(embeddings) == len(requirements_with_tiers):
    checkpoint["embeddings"] = embeddings
CHECKPOINT_PATH.write_text(json.dumps(checkpoint, indent=2), encoding="utf-8")
print(f"Saved checkpoint: {len(requirements_with_tiers)} requirements, {len(trace_pairs)} pairs.")

Saved checkpoint: 2000 requirements, 1540 pairs.
